# 01. Data Extraction & Merging
**Project:** Logistics & Delivery Delays Analysis (Olist)

## Objective
Extract and merge relevant Olist datasets, incorporating Geolocation for spatial delay analysis and filtering out irrelevant financial data.

---

## 1. Imports and Setup

In [16]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

## 2. Dataset Selection & Reasoning

The following table outlines the status of each Olist dataset relative to the **Logistics & Delivery Delays** problem statement.

| Dataset | Status | Reasoning for Inclusion/Exclusion |
| :--- | :--- | :--- |
| `olist_orders_dataset.csv` | **INCLUDED** | Core table containing order timestamps (purchase, carrier, delivery, estimation). |
| `olist_order_items_dataset.csv` | **INCLUDED** | Provides item-level details (freight value, shipping limit date) critical for logistics. |
| `olist_customers_dataset.csv` | **INCLUDED** | Contains destination location (city, state, zip) to analyze delivery distances. |
| `olist_sellers_dataset.csv` | **INCLUDED** | Contains origin location (city, state, zip) to identify shipment source. |
| `olist_products_dataset.csv` | **INCLUDED** | Physical attributes (weight, dimensions) are key drivers of shipping speed and carrier choices. |
| `olist_geolocation_dataset.csv` | **INCLUDED** | Added coordinates (Lat/Lng) to calculate displacement/distance between sellers and customers. |
| `olist_order_reviews_dataset.csv` | **INCLUDED** | Correlates delivery delays with customer satisfaction metrics. |
| `product_category_name_translation.csv` | **INCLUDED** | Required for English translation of product categories. |
| `olist_order_payments_dataset.csv` | **EXCLUDED** | **Reason:** This dataset focuses on financial transactions (installments, payment types). While interesting for revenue analysis, it does not impact physical logistics or carrier delays, which are the focus of this project. |

## 3. Loading and Pre-processing Geolocation

The geolocation dataset is aggregated to unique zip code prefixes to provide mean coordinates without row duplication.

In [17]:
PATH_RAW = "../data/raw/"

print("Loading and aggregating Geolocation...")
geo = pd.read_csv(os.path.join(PATH_RAW, 'olist_geolocation_dataset.csv'))
geo_agg = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()

print(f"Geolocation aggregated to {geo_agg.shape[0]} unique zip prefixes.")

Loading and aggregating Geolocation...
Geolocation aggregated to 19015 unique zip prefixes.


## 4. Loading Core Data & Translation

In [18]:
orders = pd.read_csv(os.path.join(PATH_RAW, 'olist_orders_dataset.csv'))
items = pd.read_csv(os.path.join(PATH_RAW, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(PATH_RAW, 'olist_customers_dataset.csv'))
sellers = pd.read_csv(os.path.join(PATH_RAW, 'olist_sellers_dataset.csv'))
products = pd.read_csv(os.path.join(PATH_RAW, 'olist_products_dataset.csv'))
reviews = pd.read_csv(os.path.join(PATH_RAW, 'olist_order_reviews_dataset.csv'))
translation = pd.read_csv(os.path.join(PATH_RAW, 'product_category_name_translation.csv'))

# Apply Translations
products = products.merge(translation, on='product_category_name', how='left')
products['product_category_name'] = products['product_category_name_english'].fillna(products['product_category_name'])
products.drop(columns=['product_category_name_english'], inplace=True)

## 5. Merging Strategy
Double joins are performed to map coordinates for both source (seller) and destination (customer).

In [19]:
df = orders.copy()
df = df.merge(items, on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(reviews, on='order_id', how='left')

# Map Customer Coordinates
df = df.merge(geo_agg, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df.rename(columns={'geolocation_lat': 'customer_lat', 'geolocation_lng': 'customer_lng'}, inplace=True)
df.drop(columns=['geolocation_zip_code_prefix'], inplace=True)

# Map Seller Coordinates
df = df.merge(geo_agg, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df.rename(columns={'geolocation_lat': 'seller_lat', 'geolocation_lng': 'seller_lng'}, inplace=True)
df.drop(columns=['geolocation_zip_code_prefix'], inplace=True)

print(f"Merge Complete: {df.shape[0]} rows and {df.shape[1]} columns.")

Merge Complete: 114092 rows and 39 columns.


## 6. Storage

In [20]:
OUTPUT_PATH = "../data/merged/olist_merged_data.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"All-in-one merged dataset saved to: {OUTPUT_PATH}")

All-in-one merged dataset saved to: ../data/merged/olist_merged_data.csv
